# VLM-DENTAL — Final paper evaluation
Author: **Reza Nadimi <rezaxr14@gmail.com>**. Built directly on **0048 / commit 57386da**; **0049 is not required**.

After SFT → GRPO: evaluate separately trained with-tools/no-tools controls; preserve the frozen primary
protocol; run selection, probes, K-sweep and inference extensions on the fixed dev split only. Nothing
here starts training, calls frontier APIs, kills unrelated processes, or pushes Git.

Outputs include raw episodes/transcripts, pooled exact/tooth F1, paired image-bootstrap intervals,
per-class and healthy-image results, format failures, localization/correction, calibration coverage,
timing, rules/skills comparisons, speculative-serving gates, LaTeX/SVG/PDF assets and a reproducibility ZIP.
Absent evidence stays pending. A notebook finishing is not a claim of diagnostic improvement.

Use an independent evaluation runtime while training is active. Start with a dev smoke run, then a full
dev run and frozen checkpoint/settings selection, then final test. Existing frontier results are imported
with their original prompts/denominators; they are never silently treated as unified local conditions.


## 1. Checkout / apply 0050
Use a checkout already carrying 0048. A fresh clone is pinned to the published 0048 commit. Upload
`0050-final-evaluation-on-0048.patch` and set `PATCH_PATH` if it has not been applied. Existing working
copies are never pulled or reset. Kaggle, Colab and local Jupyter are supported.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, importlib.util, importlib.metadata
IS_KAGGLE = Path('/kaggle/working').is_dir()
IS_COLAB = bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
WORKSPACE = Path('/kaggle/working' if IS_KAGGLE else '/content' if IS_COLAB else Path.cwd())
REPO_DIR = Path.cwd() if (Path.cwd() / 'scripts/evaluate_models.py').is_file() else WORKSPACE / 'VLM-DENTAL'
PATCH_PATH = ''  # uploaded 0050-final-evaluation-on-0048.patch
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/rezaxr14/VLM-DENTAL.git',str(REPO_DIR)],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout','57386da582177fda87191a2a28ed6e717ab7e9f6'],check=True)
os.chdir(REPO_DIR)
if PATCH_PATH:
    patch = str(Path(PATCH_PATH).expanduser().resolve())
    if subprocess.run(['git','apply','--reverse','--check',patch],capture_output=True).returncode:
        subprocess.run(['git','apply','--check',patch],check=True)
        subprocess.run(['git','apply',patch],check=True)
required = ['scripts/paper/final_eval_workflow.py','scripts/paper/generate_harness_assets.py',
            'scripts/harness/run_dev.py','scripts/bench/spec_decode_bench.py']
if any(not Path(p).is_file() for p in required):
    raise RuntimeError('Apply 0050 on top of 0048 before proceeding; 0049 is not used')
sys.path.insert(0,str(REPO_DIR))
print('Checkout:',REPO_DIR)


## 2. Experiment configuration
One profile selects precision/compute, loading, KV handling, checkpoint tags and output namespace.
All four trained controls must exist: disabling tools on a with-tools-trained checkpoint does not
create the paper's separately trained no-tools arm. Explicit local overrides are allowed and hashed.

`SMOKE_LIMIT=2` is dev-only and gets a separate output branch. Use a new `RUN_ID` and `None` for a full
run. Final test requires all twelve dev cells and the frozen selection. For CPU/TPU aggregation set
`AGGREGATE_ONLY=True`, keep the original result profile, and supply an existing phase folder containing
`run_config.json`, `run_identity.json`, `cohort.json` and episode JSONLs. No GPU/weights are needed then.


In [ ]:
PROFILE = 'dual_t4'  # dual_t4 / rtx4090 / rtxpro6000_48gb / workstation48_lora / tpu
MODE = 'dev'        # test only after section 10 freezes a complete dev selection
RUN_ID = 'dentex_paper_v1'
STAGE = 'dentex_alone'
SELECTED_K = 4
SMOKE_LIMIT = None
AGGREGATE_ONLY = False
AGGREGATE_SOURCE_DIR = ''
DATA_DIR = 'data'
DEV_SPLIT = 'data/splits/dentex_dev.json'
MODEL_ID = ''      # empty resolves attached/cache/local Qwen3.5-9B; otherwise exact local folder
HF_MODELS_REPO = 'Reza-Nadimi/vlm-dental-models'
HF_REVISION = 'main'  # resolved once to immutable SHA before checkpoint download
CHECKPOINT_OVERRIDES = {}  # local folders keyed sft_with_tools / sft_no_tools / grpo_with_tools / grpo_no_tools
GROUNDING_MODEL_PATH = ''
DENTEX_MOUNT = ''  # optional attached dataset root, otherwise reuse existing data layout
ALLOW_DOWNLOADS = True
INSTALL_DEPENDENCIES = True
INSTALL_GPU_EXTRAS = False
RUN_PRIMARY = True
RUN_PROBES = True
RUN_K_SWEEP = False
RUN_HARNESS = False
RUN_SERVING = False
RUN_MERGE = False
RUN_MTP_COMPARISON = False
BUILD_PAPER_PDF = False  # optional independent paper patch 0051 + TeX distribution required
UPLOAD_RESULTS_TO_HUB = False
RESULTS_HUB_REPO = 'Reza-Nadimi/vlm-dental-models'
CONDITIONS = ['base_no_tools','base_with_tools','sft_no_tools','sft_with_tools','grpo_no_tools','grpo_with_tools']
SIZES = ['canonical','original']
SEED = 42
N_BOOT = 2000
MAX_NEW_TOKENS = 2048
MAX_TOOL_CALLS = 50
MAX_TURNS = None
MAX_MODEL_LEN = 16384
MALFORMED_RETRIES = 0
IOU_THRESHOLD = 0.5
REMOVE_TOOLS = ['zoom_crop','window_level','locate_tooth','fdi_label','denoise','contralateral_compare','enhance_contrast','nudge_crop']
PERTURBATIONS = [(0.2,1.0),(0.5,1.0),(0.7,1.0)]
K_VALUES = [1,2,4,8,16]
K_CHECKPOINT_OVERRIDES = {}
K_LOG_TEMPLATE = ''  # optional path with literal {k}; no logs means no invented training cost
EXISTING_FRONTIER_DIR = ''
SUPERVISED_REFERENCE_JSON = ''  # AP reference kept distinct from pooled finding F1
OUTPUT_ROOT = Path('/kaggle/working/vlm_dental_final_eval' if IS_KAGGLE else WORKSPACE / 'vlm_dental_final_eval')
FROZEN_SELECTION_PATH = ''
SERVING_PYTHON = sys.executable  # point to isolated compatible serving environment
VLLM_EXECUTABLE = 'vllm'
SERVING_PORT = 8910
SERVING_GPU_INDICES = '0,1' if PROFILE == 'dual_t4' else '0'
SERVING_CONDITIONS = ['baseline','mtp_1','mtp_2','mtp_3','ngram_2','ngram_3','ngram_5']
SERVING_MEMORY_UTILIZATION = 0.9
SERVING_WARMUP_EPISODES = 1
MERGED_CHECKPOINT = ''
MTP_BEFORE_JSON = ''
MTP_AFTER_JSON = ''


## 3. Dependencies and authentication
Preserve repository dependency constraints and the runtime's PyTorch. CUDA extras are explicit and
never installed on TPU. Prepare vLLM in a separate environment using the existing speculative notebook;
this notebook records and probes its actual version rather than claiming compatibility from a flag.
Secrets are read from environment / Kaggle Secrets / Colab userdata and never printed or archived.


In [ ]:
if INSTALL_DEPENDENCIES:
    packages = [line.split('#',1)[0].strip() for line in Path('requirements.txt').read_text().splitlines()]
    packages = [p for p in packages if p and not (PROFILE == 'tpu' and p.startswith('bitsandbytes'))]
    subprocess.run([sys.executable,'-m','pip','install',*packages,'python-dotenv','nbformat','svglib','reportlab'],check=True)
    if INSTALL_GPU_EXTRAS:
        if PROFILE == 'tpu': raise ValueError('Do not install CUDA extras on TPU')
        subprocess.run([sys.executable,'-m','pip','install','-r','requirements-gpu.txt'],check=True)
from dotenv import load_dotenv
load_dotenv()
if not os.environ.get('HF_TOKEN'):
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        elif IS_COLAB:
            from google.colab import userdata
            os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
from huggingface_hub import HfApi, snapshot_download, login
if os.environ.get('HF_TOKEN'): login(token=os.environ['HF_TOKEN'],add_to_git_credential=False)
spec = importlib.util.spec_from_file_location('final_eval_workflow','scripts/paper/final_eval_workflow.py')
wf = importlib.util.module_from_spec(spec); spec.loader.exec_module(wf)
P = wf.profile(PROFILE)
RUN_ROOT = OUTPUT_ROOT / P['tag'] / STAGE / RUN_ID
BRANCH_ROOT = RUN_ROOT / ('smoke' if SMOKE_LIMIT is not None else 'full')
EVAL_DIR = BRANCH_ROOT / MODE
DEV_OUTPUT = BRANCH_ROOT / 'dev'
FREEZE_PATH = Path(FROZEN_SELECTION_PATH) if FROZEN_SELECTION_PATH else RUN_ROOT / 'freeze_selection.json'
STATUS = {key:'pending' for key in ['primary','probes','k_sweep','harness','serving','mtp_before_after']}
REMOTE_SHA = None
os.environ.setdefault('HF_ARTIFACT_REPO',HF_MODELS_REPO)
print('Profile:',P)


## 4. Exact profile-tagged checkpoints
The Hub revision is pinned once. No latest-step or other-profile fallback is made. GRPO selects the
trained `grpo_policy`, never `reference`; optimizer state is unnecessary for evaluation. All requested
controls are resolved before loading 9B weights. A missing separately trained control stops the run.


In [ ]:
adapters = {}; checkpoint_sources = {}
subfolders = wf.names(PROFILE,STAGE,SELECTED_K)
def fetch_checkpoint(condition, k=SELECTED_K, override=None):
    global REMOTE_SHA
    if override:
        return str(Path(override).expanduser().resolve()), {'source':'explicit_local'}
    if REMOTE_SHA is None:
        REMOTE_SHA = HfApi().model_info(HF_MODELS_REPO,revision=HF_REVISION).sha
    subfolder = wf.names(PROFILE,STAGE,k)[condition]
    cache = Path(snapshot_download(repo_id=HF_MODELS_REPO,revision=REMOTE_SHA,
                                  allow_patterns=[subfolder+'/*'],
                                  ignore_patterns=['**/optimizer*','**/scheduler*','**/*.pt']))
    return str(cache/subfolder), {'repo':HF_MODELS_REPO,'revision':REMOTE_SHA,'subfolder':subfolder}
if not AGGREGATE_ONLY:
    for condition in CONDITIONS:
        if condition.startswith('base_'): continue
        adapters[condition], checkpoint_sources[condition] = fetch_checkpoint(condition,override=CHECKPOINT_OVERRIDES.get(condition))
        wf.adapter_identity(adapters[condition],condition)
print(json.dumps(checkpoint_sources,indent=2))


## 5. Data / hardware / identity preflight — no 9B weights loaded
All images must resolve, including healthy controls. Dev membership is read from the committed split,
never regenerated. IDs overlap across official train/test sets, so shared loader provenance matters.

**TPU limitation:** the existing evaluator does not implement sharded multi-core autoregressive generation.
Eight 16-GB cores are not one pooled allocation for a BF16 9B target. Native generation stops before
loading on TPU; use a CUDA profile, or aggregate saved CUDA evidence on a CPU/TPU host with
`AGGREGATE_ONLY=True` and the evidence's original profile.


In [ ]:
if AGGREGATE_ONLY:
    if AGGREGATE_SOURCE_DIR:
        source = Path(AGGREGATE_SOURCE_DIR).expanduser().resolve()
        if source == EVAL_DIR.resolve() or source.is_relative_to(EVAL_DIR.resolve()) or EVAL_DIR.resolve().is_relative_to(source):
            raise ValueError('Aggregation source and destination must be disjoint')
        for path in source.rglob('*'):
            if path.is_file() and path.suffix in {'.json','.jsonl','.csv','.md','.tex','.svg','.pdf','.png','.txt','.log'}:
                target = EVAL_DIR / path.relative_to(source); target.parent.mkdir(parents=True,exist_ok=True)
                if target.exists() and wf.digest(target) != wf.digest(path): raise ValueError('Import would replace evidence: '+str(target))
                shutil.copy2(path,target)
    CFG = json.loads((EVAL_DIR/'run_config.json').read_text())
    IDENTITY = json.loads((EVAL_DIR/'run_identity.json').read_text())
    EXPECTED_IDS = json.loads((EVAL_DIR/'cohort.json').read_text())['image_ids']
    if CFG['mode'] != MODE or wf.profile(CFG['profile'])['tag'] != P['tag']: raise ValueError('Select the original result profile/split')
    CFG['output'] = str(EVAL_DIR); CFG['python'] = sys.executable
else:
    import torch, platform
    from argparse import Namespace
    from scripts import evaluate_models as ev
    from dental_agent.utils.inputs import resolve_qwen_model, resolve_yolo_checkpoint, LOCAL_BASE_MODEL_ENV
    from dental_agent.data.dev_split import assert_not_test_split
    from dental_agent.data.dentex import find_local_dentex_dir
    if P['name'] == 'tpu':
        raise RuntimeError('No sharded 9B TPU generation in this evaluator. Use aggregation-only or a CUDA evaluation runtime; weights were not loaded.')
    if not torch.cuda.is_available(): raise RuntimeError('Selected native evaluation profile requires CUDA')
    devices = [{'name':torch.cuda.get_device_name(i),'gib':torch.cuda.get_device_properties(i).total_memory/2**30,
                'capability':torch.cuda.get_device_capability(i)} for i in range(torch.cuda.device_count())]
    if P['name'] == 'dual_t4' and (len(devices)!=2 or any('T4' not in d['name'] for d in devices)): raise ValueError('dual_t4 does not match detected hardware')
    if P['name'] == 'rtx4090' and (len(devices)!=1 or '4090' not in devices[0]['name']): raise ValueError('rtx4090 does not match detected hardware')
    if P['name'] == 'workstation48_lora' and (len(devices)!=1 or devices[0]['gib']<44 or not torch.cuda.is_bf16_supported()): raise ValueError('48-GB BF16 profile does not match detected hardware')
    if DENTEX_MOUNT:
        mount=Path(DENTEX_MOUNT).expanduser().resolve()
        if not mount.is_dir(): raise FileNotFoundError(mount)
        link=Path(DATA_DIR)/'dentex/images'; link.parent.mkdir(parents=True,exist_ok=True)
        if not link.exists(): link.symlink_to(mount/'images' if (mount/'images').is_dir() else mount,target_is_directory=True)
    if not ALLOW_DOWNLOADS:
        for split in ({'train','test'} if MODE=='test' else {'train'}):
            if find_local_dentex_dir(DATA_DIR,split) is None: raise RuntimeError('Missing local '+split+' annotations; attach data or explicitly allow downloads')
    images, annotations, categories = ev.load_eval_split(Namespace(split=MODE,dataset='dentex',dev_split=DEV_SPLIT,data_dir=DATA_DIR))
    if MODE=='dev': assert_not_test_split(images)
    elif len(images)!=50: raise ValueError('Final cohort must contain all 50 official test images')
    missing=images[~images['local_path'].apply(lambda p:isinstance(p,str) and Path(p).is_file())]
    if len(missing): raise FileNotFoundError(f'{len(missing)} image paths unresolved; do not shrink final denominators')
    EXPECTED_IDS=[int(i) for i in ev.eligible_images(images,SMOKE_LIMIT)['id']]
    if not EXPECTED_IDS: raise ValueError('Empty cohort')
    yolo=resolve_yolo_checkpoint(GROUNDING_MODEL_PATH or None,repo=HF_MODELS_REPO,hub='download' if ALLOW_DOWNLOADS else 'off')
    os.environ['GROUNDING_MODEL_PATH']=str(yolo.path.resolve())
    if MODEL_ID: os.environ[LOCAL_BASE_MODEL_ENV]=str(Path(MODEL_ID).expanduser().resolve())
    MODEL_ID,MODEL_SOURCE=resolve_qwen_model(download=ALLOW_DOWNLOADS)
    CFG=dict(profile=PROFILE,mode=MODE,run_id=RUN_ID,stage=STAGE,selected_k=SELECTED_K,smoke_limit=SMOKE_LIMIT,
             conditions=CONDITIONS,sizes=SIZES,model_id=MODEL_ID,adapters=adapters,output=str(EVAL_DIR),python=sys.executable,
             data_dir=str(Path(DATA_DIR).resolve()),dev_split=str(Path(DEV_SPLIT).resolve()),seed=SEED,n_boot=N_BOOT,
             max_new_tokens=MAX_NEW_TOKENS,max_tool_calls=MAX_TOOL_CALLS,max_turns=MAX_TURNS,
             malformed_retries=MALFORMED_RETRIES,iou_threshold=IOU_THRESHOLD)
    wf.validate(CFG)
    identities={c:wf.adapter_identity(path,c,CFG) for c,path in adapters.items()}
    versions={}
    for package in ['torch','transformers','peft','accelerate','bitsandbytes','numpy','scikit-learn','ultralytics']:
        try: versions[package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: versions[package]=None
    base=Path(MODEL_ID)
    IDENTITY={'adapters':{c:v['files'] for c,v in identities.items()},'grounding_sha256':wf.digest(yolo.path),
              'dev_split_sha256':wf.digest(DEV_SPLIT),'code_sha256':wf.fingerprint(wf.source_files()),'software':versions,
              'base_metadata':{p.name:wf.digest(p) for p in base.iterdir() if p.name in {'config.json','model.safetensors.index.json','tokenizer_config.json','preprocessor_config.json'}},
              'base_shard_sizes':{p.name:p.stat().st_size for p in base.glob('*.safetensors')},'base_all_weight_bytes_hashed':False}
    if MODE=='test': wf.verify_frozen(FREEZE_PATH,CFG,IDENTITY)
    wf.lock(EVAL_DIR/'run_identity.json',IDENTITY); wf.lock(EVAL_DIR/'run_config.json',CFG)
    cohort={'split':MODE,'image_ids':EXPECTED_IDS,'images':[]}
    for row in images[images['id'].isin(EXPECTED_IDS)].to_dict('records'):
        cohort['images'].append({'id':int(row['id']),'path':str(row['local_path']),'source_file':str(row.get('source_file')),'sha256':wf.digest(row['local_path'])})
    wf.lock(EVAL_DIR/'cohort.json',cohort)
    wf.write(EVAL_DIR/'checkpoint_metadata.json',identities)
    wf.write(EVAL_DIR/'checkpoint_sources.json',checkpoint_sources)
    wf.write(EVAL_DIR/'environment.json',{'python':sys.version,'platform':platform.platform(),'packages':versions,'devices':devices,
                                      'git_head':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()})
print(f'Preflight: {len(EXPECTED_IDS)} {MODE} images. No 9B model weights have been loaded by this notebook yet.')


## 6. Primary matrix
Base groups share a load; each trained track is a separate invocation with its own adapter. The same
shared runner performs greedy generation and writes each episode incrementally. Existing matching
files resume; cached legacy zero-shot prompts are not reused as unified-prompt results.


In [ ]:
import shlex
commands=[] if AGGREGATE_ONLY else wf.primary_commands(CFG)
for command in commands: print(shlex.join(command))
if RUN_PRIMARY and not AGGREGATE_ONLY:
    wf.run(commands,EVAL_DIR/'logs/primary')
    done=wf.completion(CFG,EXPECTED_IDS)
    wf.write(EVAL_DIR/'primary_completion.json',done)
    STATUS['primary']='complete' if SMOKE_LIMIT is None else 'dev_smoke_only'


## 7. Dev-only tool removal and grounding correction
Remove each diagnostic tool from the registry/prompt and retain attempts to call it. This measures
tool dependence, not separately trained no-tool performance. Grounding probes shift the same image/tooth
box deterministically across SFT/GRPO; reports retain initial/final IoU, crop coverage, successes and
failures. These are dev-only. No test-set tuning or silent reward/prompt changes occur.


In [ ]:
if MODE=='dev' and RUN_PROBES and not AGGREGATE_ONLY:
    commands=wf.probe_commands(CFG,REMOVE_TOOLS,PERTURBATIONS)
    wf.write(EVAL_DIR/'probe_commands.json',commands)
    wf.run(commands,EVAL_DIR/'logs/probes')
    from dental_agent.evaluation.matrix import discover_ablation_tags,cell_path
    checked={}
    for tag in discover_ablation_tags(EVAL_DIR,'dentex','dev'):
        for condition in [c for c in CONDITIONS if c.endswith('with_tools') and not c.startswith('base_')]:
            path=cell_path(EVAL_DIR,'dentex','dev',condition,'canonical',tag)
            wf.cohort(wf.strict_records(path),EXPECTED_IDS,'dev');checked[str(path)]=wf.digest(path)
    wf.write(EVAL_DIR/'probe_completion.json',checked)
    STATUS['probes']='complete' if SMOKE_LIMIT is None else 'dev_smoke_only'


## 8. Already-trained K-sweep and training cost (dev only)
The sweep evaluates existing K checkpoints and reports paired comparisons and available training logs.
It does not train or select a winning K automatically. After choosing K on dev, rerun its full primary
dev matrix under a new run ID before freezing. Missing training cost is not zero.


In [ ]:
if MODE=='dev' and RUN_K_SWEEP and not AGGREGATE_ONLY:
    sweep=BRANCH_ROOT/'k_sweep'
    for k in K_VALUES:
        adapter,source=fetch_checkpoint('grpo_with_tools',k, K_CHECKPOINT_OVERRIDES.get(k))
        kc=copy_cfg=json.loads(json.dumps(CFG))
        kc.update(selected_k=k,conditions=['grpo_with_tools'],sizes=['canonical'],output=str(sweep/f'k{k}'))
        kc['adapters']['grpo_with_tools']=adapter
        ki=wf.adapter_identity(adapter,'grpo_with_tools',kc)
        wf.lock(Path(kc['output'])/'run_identity.json',{'adapter':ki['files'],'run':IDENTITY,'source':source})
        wf.lock(Path(kc['output'])/'run_config.json',kc)
        wf.run([wf.matrix_command(kc,['grpo_with_tools'])],sweep/'logs')
        wf.completion(kc,EXPECTED_IDS)
    command=[sys.executable,'scripts/aggregate_k_sweep.py','--sweep-dir',str(sweep),'--reference-dir',str(DEV_OUTPUT),
             '--split','dev','--size','canonical','--n-boot',str(N_BOOT),'--seed',str(SEED)]
    if K_LOG_TEMPLATE:command+=['--log-template',K_LOG_TEMPLATE]
    wf.run([command],sweep/'logs');STATUS['k_sweep']='complete' if SMOKE_LIMIT is None else 'dev_smoke_only'


## 9. Shared metrics, paired intervals, calibration and publication figures
The existing aggregator recomputes set matching and pools finding counts before F1, resampling whole
images in paired comparisons. It exports per-class/pathological-only performance, healthy-image
invention, format failures, complete-case agreement, costs and localization. Canonical/original views
remain distinct. Supplementary finding calibration excludes missing/invalid confidences and retains
coverage/raw pairs; it uses the shared ECE, including its documented zero-confidence bin limitation.

The LaTeX primary table is a **candidate fragment** for review. Existing frontier results are imported
without API calls under their original prompt/denominator. Specialist AP is not relabeled finding F1.


In [ ]:
from IPython.display import display,Markdown
if any(EVAL_DIR.glob(f'eval_dentex_{MODE}_*.jsonl')):
    for path in EVAL_DIR.glob(f'eval_dentex_{MODE}_*.jsonl'): wf.strict_records(path)
    shared=['--output-dir',str(EVAL_DIR),'--dataset','dentex','--split',MODE]
    wf.run([[sys.executable,'scripts/aggregate_eval.py',*shared,'--n-boot',str(N_BOOT),'--seed',str(SEED),'--iou-threshold',str(IOU_THRESHOLD)],
            [sys.executable,'scripts/plot_eval_figures.py',*shared,'--formats','png','pdf','svg']],EVAL_DIR/'logs/reporting')
    matrix_dir=EVAL_DIR/f'matrix_dentex_{MODE}'
    summary=json.loads((matrix_dir/'matrix_summary.json').read_text())
    wf.calibration(EVAL_DIR,MODE,matrix_dir/'finding_calibration.json')
    wf.primary_tex(summary,matrix_dir/'table_primary_candidate.tex')
    display(Markdown((matrix_dir/'matrix_summary.md').read_text()))
    if MODE=='test':
        wf.write(EVAL_DIR/'final_test_completion.json',wf.completion(CFG,EXPECTED_IDS));STATUS['primary']='complete'
for label,source in [('frontier',EXISTING_FRONTIER_DIR),('specialist',SUPERVISED_REFERENCE_JSON)]:
    if not source:continue
    source=Path(source).expanduser().resolve()
    if not source.exists():raise FileNotFoundError(source)
    target_dir=RUN_ROOT/'existing_references'/label; imported=[]
    for path in (source.rglob('*') if source.is_dir() else [source]):
        if path.is_file() and path.suffix in {'.json','.jsonl','.csv','.md','.tex'}:
            relative=path.relative_to(source) if source.is_dir() else Path(path.name)
            target=target_dir/relative;target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists() and wf.digest(target)!=wf.digest(path):raise ValueError('Reference import changed; use a new run ID')
            shutil.copy2(path,target);imported.append({'file':str(relative),'sha256':wf.digest(target)})
    wf.write(target_dir/'provenance.json',{'source':str(source),'files':imported,'note':'Original prompt/denominator or specialist AP; not a paired unified local cell'})


## 10. Freeze selection → final test
A full twelve-cell dev run records exact checkpoint/code/data/runtime/budget identity. A smoke run
cannot freeze a final selection. To evaluate the final test, switch `MODE='test'` in section 2 and
rerun with the same profile, stage, selected K and run ID. A changed identity is rejected. This is an
experimental lock, not retrospective preregistration or proof of missing historical training provenance.


In [ ]:
if MODE=='dev' and SMOKE_LIMIT is None and RUN_PRIMARY and not AGGREGATE_ONLY:
    wf.freeze(FREEZE_PATH,CFG,IDENTITY,wf.completion(CFG,EXPECTED_IDS))
    print('Frozen selection:',FREEZE_PATH)
elif MODE=='test' and not AGGREGATE_ONLY:
    wf.verify_frozen(FREEZE_PATH,CFG,IDENTITY)


## 11. Rules and skills — paired dev-only inference extension
Evaluate baseline, exact rules, index/on-demand and full injection on a fixed with-tools checkpoint.
Rules-only is a prompt-identity control. The runner records skill names, versions, hashes and disclosure
events. Finding invention `FP/(TP+FP)` differs from the primary healthy-image invention metric.
Partial harness runs are retained, not merged manually into a completed comparison.


In [ ]:
HARNESS_JSON=BRANCH_ROOT/'extensions/harness_dev.json'
HARNESS_KIND='grpo'  # sft or grpo
if MODE=='dev' and RUN_HARNESS and not AGGREGATE_ONLY:
    wf.lock(HARNESS_JSON.with_suffix('.identity.json'),{'run':IDENTITY,'kind':HARNESS_KIND,'smoke_limit':SMOKE_LIMIT})
    if HARNESS_JSON.exists():
        if json.loads(HARNESS_JSON.read_text()).get('status')!='complete':raise ValueError('Retain partial harness evidence and use a new run ID')
    else:
        command=[sys.executable,'-u','scripts/harness/run_dev.py','--model-id',MODEL_ID,
                 '--adapter-path',adapters[HARNESS_KIND+'_with_tools'],'--checkpoint-kind',HARNESS_KIND,
                 '--data-dir',DATA_DIR,'--dev-split',DEV_SPLIT,'--split','dev','--conditions',
                 'baseline','rules_only','rules_skills_index','rules_skills_full','--output',str(HARNESS_JSON),
                 '--seed',str(SEED),'--max-new-tokens',str(MAX_NEW_TOKENS),'--max-tool-calls',str(MAX_TOOL_CALLS),*wf.runtime_flags(PROFILE)]
        if MAX_TURNS is not None:command+=['--max-turns',str(MAX_TURNS)]
        if SMOKE_LIMIT is not None:command+=['--limit',str(SMOKE_LIMIT)]
        wf.run([command],BRANCH_ROOT/'extensions/logs')
    STATUS['harness']='complete' if SMOKE_LIMIT is None else 'dev_smoke_only'


## 12. Serving environment, CPU merge and MTP byte audit
Use a separate compatible vLLM environment (existing speculative notebook provides setup). A merged
16-bit target differs numerically from 4-bit QLoRA inference; baseline/speculation must share that same
merged target. CPU merging writes a new folder and needs sufficient host RAM/disk. A real MTP tensor
key/shape/dtype/content audit is required; preserved tensors do not by themselves guarantee acceptance.


In [ ]:
SERVING_DIR=BRANCH_ROOT/'extensions/serving'
SERVING_JSON=SERVING_DIR/'spec_decode_bench.json'
if not MERGED_CHECKPOINT:MERGED_CHECKPOINT=str(BRANCH_ROOT/'merged_target')
if MODE=='dev' and (RUN_SERVING or RUN_MERGE) and not AGGREGATE_ONLY:
    if P['name']=='tpu':raise ValueError('CUDA serving only')
    wf.run([[SERVING_PYTHON,'scripts/bench/probe_environment.py','--vllm-executable',VLLM_EXECUTABLE,
             '--gpu-indices',SERVING_GPU_INDICES,'--output',str(SERVING_DIR/'environment_probe.json')]],SERVING_DIR/'logs')
    if RUN_MERGE:
        wf.run([[SERVING_PYTHON,'scripts/bench/merge_adapter.py','--base',MODEL_ID,
                 '--adapter',identities['grpo_with_tools']['policy_path'],'--output',MERGED_CHECKPOINT]],SERVING_DIR/'logs')
    if not Path(MERGED_CHECKPOINT).is_dir():raise FileNotFoundError('Supply an audited merged target or enable explicit merge')
    wf.run([[SERVING_PYTHON,'scripts/bench/check_mtp.py','--base',MODEL_ID,'--checkpoint',MERGED_CHECKPOINT,
             '--output',str(SERVING_DIR/'mtp_audit.json')]],SERVING_DIR/'logs')


## 13. Same-target baseline / MTP / lookup benchmark (dev only)
Owned servers run serially; none of your other jobs are terminated. Same images, greedy settings,
prompts and tools; requested profiles are T4 FP16/TP2 and 4090/48-GB BF16/TP1. Real startup/equivalence
must establish support and fit. Context headroom is never silently reduced. Failed modes retain logs.
Decoded text/actions/final answers and dev F1 must agree; token-ID or stochastic equivalence is not
claimed. Missing prefill/decode/acceptance counters stay null; TTFT is not renamed prefill.


In [ ]:
if MODE=='dev' and RUN_SERVING and not AGGREGATE_ONLY:
    if SERVING_JSON.exists():raise ValueError('Serving evidence already exists; retain it and use a new run ID')
    devices=[x.strip() for x in SERVING_GPU_INDICES.split(',') if x.strip()]
    command=[SERVING_PYTHON,'-u','scripts/bench/spec_decode_bench.py','--checkpoint',MERGED_CHECKPOINT,
             '--data-dir',DATA_DIR,'--dev-split',DEV_SPLIT,'--split','dev','--conditions',*SERVING_CONDITIONS,
             '--output',str(SERVING_JSON),'--vllm-executable',VLLM_EXECUTABLE,'--gpu-indices',SERVING_GPU_INDICES,
             '--tensor-parallel',str(len(devices)),'--dtype','float16' if P['name']=='dual_t4' else 'bfloat16',
             '--max-model-len',str(MAX_MODEL_LEN),'--max-new-tokens',str(MAX_NEW_TOKENS),
             '--max-tool-calls',str(MAX_TOOL_CALLS),'--seed',str(SEED),'--port',str(SERVING_PORT),
             '--warmup-episodes',str(SERVING_WARMUP_EPISODES),'--gpu-memory-utilization',str(SERVING_MEMORY_UTILIZATION)]
    if MAX_TURNS is not None:command+=['--max-turns',str(MAX_TURNS)]
    if SMOKE_LIMIT is not None:command+=['--limit',str(SMOKE_LIMIT)]
    wf.run([command],SERVING_DIR/'logs')
    STATUS['serving']=json.loads(SERVING_JSON.read_text())['status']


## 14. Optional measured MTP before/after comparison
Read completed benchmark files from the companion MTP-only training workflow. No MTP training starts
here. Keep on-policy train-minus-dev regeneration provenance and the unchanged non-MTP tensor audit
with the paper evidence. The comparator requires shared dev settings, outputs and valid gates.


In [ ]:
if MODE=='dev' and RUN_MTP_COMPARISON and not AGGREGATE_ONLY:
    if not MTP_BEFORE_JSON or not MTP_AFTER_JSON:raise ValueError('Supply measured before/after JSON')
    wf.run([[SERVING_PYTHON,'scripts/bench/compare_acceptance.py','--before',MTP_BEFORE_JSON,
             '--after',MTP_AFTER_JSON,'--output',str(BRANCH_ROOT/'extensions/mtp_before_after.json')]],BRANCH_ROOT/'extensions/logs')
    for name,path in [('before',MTP_BEFORE_JSON),('after',MTP_AFTER_JSON)]:
        shutil.copy2(path,BRANCH_ROOT/'extensions'/('mtp_'+name+'.json'))
    STATUS['mtp_before_after']='complete'


## 15. LaTeX assets, measured plots and optional PDF
0050 includes its result validator/exporter directly on 0048. Complete compatible dev evidence can
fill cells; missing experiments/counters stay pending. Smoke runs cannot populate manuscript results.
The independent replacement paper patch 0051 supplies detailed diagrams, measured plots and the clean
PDF builder; 0049 is never used. Primary tables remain reviewable candidates rather than automatic
edits to the frozen manuscript table.


In [ ]:
PAPER_ASSETS=BRANCH_ROOT/'paper_assets';PAPER_ASSETS.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'scripts/paper/generate_harness_assets.py','--output-dir',str(PAPER_ASSETS)]
usable={}
for flag,path in [('--harness-results',HARNESS_JSON),('--serving-results',SERVING_JSON)]:
    if path.is_file() and SMOKE_LIMIT is None:
        data=json.loads(path.read_text());status=data.get('status')
        if status=='complete' or (flag=='--serving-results' and status=='failed_or_divergent' and all(v.get('status')=='complete' for v in data.get('conditions',{}).values())):
            command += [flag,str(path)];usable[flag]=path
        else:print(path.name,status,'retained; table pending')
wf.run([command],BRANCH_ROOT/'logs/paper')
if usable and Path('scripts/paper/plot_harness_serving.py').is_file():
    plot=[sys.executable,'scripts/paper/plot_harness_serving.py','--paper-dir',str(PAPER_ASSETS)]
    for flag,path in usable.items():plot += [flag,str(path)]
    wf.run([plot],BRANCH_ROOT/'logs/paper')
if BUILD_PAPER_PDF:
    if SMOKE_LIMIT is not None:raise ValueError('Smoke data cannot populate the paper')
    if not Path('scripts/paper/build_paper.py').is_file():raise ValueError('Apply independent paper patch 0051 for PDF builds')
    for name in ['table_harness_dev.tex','table_serving_efficiency.tex','harness_evidence.json']:
        shutil.copy2(PAPER_ASSETS/name,Path('paper/generated')/name)
    # Optional measured figure assets, if produced, are copied with their generated fragment.
    for source in [PAPER_ASSETS/'generated/harness_serving_plot.tex',PAPER_ASSETS/'generated/harness_serving_plot_manifest.json']:
        if source.is_file():shutil.copy2(source,Path('paper/generated')/source.name)
    for kind in ['svg','pdf']:
        source=PAPER_ASSETS/f'figures/{kind}/fig_harness_serving_results.{kind}'
        if source.is_file():shutil.copy2(source,Path(f'paper/figures/{kind}')/source.name)
    wf.run([[sys.executable,'scripts/paper/verify_paper.py'],[sys.executable,'scripts/paper/build_paper.py',
             '--output-dir',str(PAPER_ASSETS/'pdf_build')]],BRANCH_ROOT/'logs/paper')


## 16. Evidence status, archive and download
The ZIP contains episodes/transcripts, commands/logs, checkpoint/data/code identities, tables/figures,
calibration coverage, extension evidence and explicit pending/failure status. It excludes model weights,
optimizer state and secrets. Export these cells manually after a failed run to salvage completed
JSONLs without rerunning inference. Kaggle: download from Outputs under `/kaggle/working/`.
The source snapshot and original notebook are included so a remote branch is not the only record.


In [ ]:
import zipfile,datetime
if not AGGREGATE_ONLY and wf.fingerprint(wf.source_files())!=IDENTITY['code_sha256']:
    raise ValueError('Code changed during evaluation; preserve raw evidence and use a new run identity')
if (EVAL_DIR/'primary_completion.json').is_file() or (EVAL_DIR/'final_test_completion.json').is_file():
    wf.completion(CFG,EXPECTED_IDS);STATUS['primary']='complete' if SMOKE_LIMIT is None else 'dev_smoke_only'
for name,path in [('harness',HARNESS_JSON),('serving',SERVING_JSON)]:
    if path.is_file():STATUS[name]=json.loads(path.read_text()).get('status','unknown')
if (DEV_OUTPUT/'probe_completion.json').is_file():STATUS['probes']='recorded; see raw cohort audit'
if SMOKE_LIMIT is not None:STATUS={k:'dev_smoke_only/'+v if v!='pending' else v for k,v in STATUS.items()}
report={'mode':MODE,'profile':P,'stage':STAGE,'selected_k':SELECTED_K,'expected_images':len(EXPECTED_IDS),
        'aggregation_only':AGGREGATE_ONLY,'status':STATUS,'analysis_code_sha256':wf.fingerprint(wf.source_files()),
        'recorded_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'notes':['Finding false discovery differs from healthy-image invention.',
                 'Base identity hashes metadata/index and shard sizes, not every base tensor byte.',
                 'Training metadata gaps are recorded, not fabricated.',
                 'Tufts used in training is not an untouched external test set.',
                 'Clinical-task improvement, speed, memory fit and hardware support need real measurements.']}
wf.write(BRANCH_ROOT/'evidence_status.json',report)
ARCHIVE=RUN_ROOT/f'final_eval_{P["tag"]}_{STAGE}_{RUN_ID}.zip';ARCHIVE.parent.mkdir(parents=True,exist_ok=True)
allowed={'.json','.jsonl','.csv','.md','.tex','.svg','.pdf','.png','.txt','.log','.ipynb'}
files=[p for p in RUN_ROOT.rglob('*') if p.is_file() and p.suffix in allowed and not any(x in {'merged_target','hidden_states','checkpoints'} for x in p.relative_to(RUN_ROOT).parts)]
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(files):archive.write(path,path.relative_to(RUN_ROOT))
    archive.write('notebooks/VLM_Dental_Colab_Final_Evaluation.ipynb','notebook/VLM_Dental_Colab_Final_Evaluation.ipynb')
    for relative in wf.source_files():archive.write(REPO_DIR/relative,'analysis_source/'+relative)
print(json.dumps(report,indent=2));print('Archive:',ARCHIVE)


In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))
else:
    from IPython.display import FileLink,display
    display(FileLink(str(ARCHIVE)))
    print('Kaggle: use Outputs to download the ZIP at the printed path.')


## 17. Optional evidence upload
Off by default. Enabling this uploads only the evidence ZIP into a profile/stage/run-specific evaluation
namespace on your Hub repository; it never replaces model checkpoints or pushes Git.


In [ ]:
if UPLOAD_RESULTS_TO_HUB:
    target=f'evaluations/final/{P["tag"]}/{STAGE}/{RUN_ID}/{ARCHIVE.name}'
    result=HfApi().upload_file(path_or_fileobj=str(ARCHIVE),repo_id=RESULTS_HUB_REPO,repo_type='model',
                             path_in_repo=target,commit_message=f'Evaluation evidence: {P["tag"]}/{STAGE}/{RUN_ID}')
    print('Evidence commit:',result)
else:print('Hub upload disabled')
